In [15]:
from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

data = load_iris()

X = data.data
y = data.target


# --------------------------------------------------
# 2. Hold out final test set
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# --------------------------------------------------
# 3. Define candidate models
# --------------------------------------------------

models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000
        ))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]),

    "Decision Tree":
        DecisionTreeClassifier(
            max_depth=4,
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
}


# --------------------------------------------------
# 4. 5-fold cross-validation
# --------------------------------------------------

results = {}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    results[name] = {
        "mean": scores.mean(),
        "std": scores.std()
    }


# --------------------------------------------------
# 5. Display results
# --------------------------------------------------

for name, result in results.items():

    print(
        f"{name:20} "
        f"Mean = {result['mean']:.4f}, "
        f"Std = {result['std']:.4f}"
    )

Logistic Regression  Mean = 0.9583, Std = 0.0264
KNN                  Mean = 0.9667, Std = 0.0312
Decision Tree        Mean = 0.9417, Std = 0.0204
Random Forest        Mean = 0.9500, Std = 0.0167
Gradient Boosting    Mean = 0.9667, Std = 0.0167
XGBoost              Mean = 0.9500, Std = 0.0167


In [16]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

lr_grid = GridSearchCV(
    estimator=models['Logistic Regression'],
    param_grid=lr_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

print("Best parameters:", lr_grid.best_params_)
print("Best CV score:", lr_grid.best_score_)

Best parameters: {'model__C': 10}
Best CV score: 0.9666666666666668


In [17]:
gb_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

gb_grid = GridSearchCV(
    estimator=models['Gradient Boosting'],
    param_grid=gb_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)

print("Best parameters:", gb_grid.best_params_)
print("Best CV score:", gb_grid.best_score_)

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best CV score: 0.9666666666666668


In [18]:
rf_params = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

rf_random = RandomizedSearchCV(
    estimator=models['Random Forest'],
    param_distributions=rf_params,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_train, y_train)

print("Best parameters:", rf_random.best_params_)
print("Best CV score:", rf_random.best_score_)

Best parameters: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Best CV score: 0.9583333333333334


In [19]:
xgb_params = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5, 6],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

xgb_random = RandomizedSearchCV(
    estimator=models['XGBoost'],
    param_distributions=xgb_params,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

xgb_random.fit(X_train, y_train)

print("Best parameters:", xgb_random.best_params_)
print("Best CV score:", xgb_random.best_score_)

Best parameters: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Best CV score: 0.9583333333333334


In [20]:
tuned_results = {
    "Logistic Regression": lr_grid.best_score_,
    "Gradient Boosting": gb_grid.best_score_,
    "Random Forest": rf_random.best_score_,
    "XGBoost": xgb_random.best_score_,
    "KNN (untuned)": results["KNN"]["mean"]
}

for name, score in sorted(
    tuned_results.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{name:25} {score:.4f}")

Logistic Regression       0.9667
Gradient Boosting         0.9667
KNN (untuned)             0.9667
Random Forest             0.9583
XGBoost                   0.9583


In [21]:
print("LR:", lr_grid.best_params_, lr_grid.best_score_)
print("GB:", gb_grid.best_params_, gb_grid.best_score_)
print("RF:", rf_random.best_params_, rf_random.best_score_)
print("XGB:", xgb_random.best_params_, xgb_random.best_score_)

LR: {'model__C': 10} 0.9666666666666668
GB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100} 0.9666666666666668
RF: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None} 0.9583333333333334
XGB: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.7} 0.9583333333333334


In [22]:
best_model = gb_grid.best_estimator_

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print("Final Test Accuracy:", test_accuracy)

Final Test Accuracy: 0.9666666666666667


In [23]:
finalists = {
    "Logistic Regression": lr_grid.best_estimator_,
    "Gradient Boosting": gb_grid.best_estimator_,
    "KNN": models['KNN']
}

for name, model in finalists.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    score = accuracy_score(y_test, y_pred)

    print(f"{name:25} Test Accuracy = {score:.4f}")

Logistic Regression       Test Accuracy = 1.0000
Gradient Boosting         Test Accuracy = 0.9667
KNN                       Test Accuracy = 0.9333


Trying KNN tuning as skipped before for high sd

In [24]:
knn_params = {
    "model__n_neighbors": [1, 3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"]
}

knn_grid = GridSearchCV(
    estimator=models['KNN'],
    param_grid=knn_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

print("Best parameters:", knn_grid.best_params_)
print("Best CV score:", knn_grid.best_score_)

Best parameters: {'model__metric': 'euclidean', 'model__n_neighbors': 5, 'model__weights': 'uniform'}
Best CV score: 0.9666666666666668


In [25]:
final_cv_results = {
    "Logistic Regression": lr_grid.best_score_,
    "Gradient Boosting": gb_grid.best_score_,
    "KNN": knn_grid.best_score_,
    "Random Forest": rf_random.best_score_,
    "XGBoost": xgb_random.best_score_
}

for name, score in sorted(
    final_cv_results.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{name:25} {score:.4f}")

Logistic Regression       0.9667
Gradient Boosting         0.9667
KNN                       0.9667
Random Forest             0.9583
XGBoost                   0.9583
